This notebook demonstrates how to configure, inspect and execute a semantic segmentation experiment using the GeoTest framework.

The example uses CROMA Large with a frozen encoder and an FCN segmentation head on the Sen1Floods11 dataset, combining Sentinel-1 and Sentinel-2 imagery.

Training and evaluation are performed through the shared run.py experiment runner to ensure consistency with the main framework.
>**Note**: This notebook is intended as a demonstration of the experimental pipeline, not as a reproduction of the final benchmarking results. Running the training cell starts a new experiment.

#### Imports and project directory

In [ ]:
import os
import sys
from pathlib import Path

project_dir = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "run.py").is_file() and (path / "configs").is_dir()),
    None,
)
if project_dir is None:
    raise FileNotFoundError("Open this notebook from GeoTest or its experiments directory.")
os.chdir(project_dir)
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

import json
import torch
from IPython.display import Image, display


In [ ]:
from run import run_experiment
from utils import build_dataset, load_config


#### Read and validate the YAML configuration

In [ ]:
config_path = Path("configs/croma/fcn/croma_s1s2_fcn_all_hand_labels.yaml")
config = load_config(config_path, show_config=True)


In [ ]:
data_cfg = config["dataset"]
model_cfg = config["model"]
train_cfg = config["training"]
eval_cfg = config["evaluation"]
preprocess_cfg = config["preprocessing"]

print("Experiment:", config["experiment"]["name"])
print("Dataset:", data_cfg["name"])
print("Modalities:", data_cfg["modalities"])


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Seed:", config["experiment"].get("seed", 42))
# The runner initializes independent model, shuffle, crop and loader streams.


In [ ]:
modalities = tuple(data_cfg["modalities"])
print("Model:", model_cfg["name"])
print("Training crop size:", preprocess_cfg["crop_size"])
print("Evaluation:", json.dumps(eval_cfg, indent=2))


In [ ]:
split_paths = {
    split: Path(data_cfg[f"{split}_csv"])
    for split in ("train", "val", "test", "bolivia")
    if data_cfg.get(f"{split}_csv")
}
for split, path in split_paths.items():
    print(f"{split}: {path}")


#### Inspect dataset membership and check the splits

In [ ]:
raw_datasets = {
    split: build_dataset(config, split)
    for split in split_paths
}
seen = {}
for split, dataset in raw_datasets.items():
    print(f"{split}: {len(dataset)} chips before training exclusions")
    for paths in dataset.data_list:
        chip = paths["target"].name
        if chip in seen:
            raise ValueError(f"Chip appears in {seen[chip]} and {split}: {chip}")
        seen[chip] = split

# Membership inspection does not load image arrays or consume crop randomness.
del raw_datasets


#### Inspect normalization and band selection

In [ ]:
print("Normalization:", json.dumps(preprocess_cfg["normalization"], indent=2))
print("Band indices:", json.dumps(preprocess_cfg.get("band_indices", {}), indent=2))
# The runner computes or loads training statistics, or selects fixed statistics.


#### Inspect training crops and DataLoader settings

In [ ]:
print("Training batch size:", train_cfg["batch_size"])
print("Evaluation image batch size:", eval_cfg.get("batch_size", 1))
print("Evaluation tile batch size:", eval_cfg.get("tile_batch_size", 1))
print("DataLoader workers:", train_cfg.get("num_workers", 0))
print("Entirely ignored training chips are excluded; crops require valid pixels.")
print("Valid crops without water are retained.")


#### Inspect the encoder or supervised baseline configuration

In [ ]:
print(json.dumps(model_cfg, indent=2))
# Model construction is deferred to the training cell to avoid duplicate encoders.


#### Inspect the adapter and segmentation head

In [ ]:
print("Adapter:", json.dumps(config.get("adapter", {}), indent=2))
print("Head:", json.dumps(config["head"], indent=2))


#### Inspect the loss, optimizer, scheduler and checkpoint selection

In [ ]:
print("Epochs:", train_cfg["epochs"])
print("Validate every:", train_cfg.get("validate_every", 5), "epochs")
print("Loss:", json.dumps(train_cfg["loss"], indent=2))
print("Optimizer:", json.dumps(train_cfg["optimizer"], indent=2))
print("Scheduler:", json.dumps(train_cfg.get("scheduler") or {"name": "none"}, indent=2))
print("Checkpoint metric:", eval_cfg.get("checkpoint_metric", "miou"))
if eval_cfg.get("checkpoint_metric", "miou") == "iou":
    print("Checkpoint class:", eval_cfg.get("checkpoint_class", 1))


#### Inspect the output location and held-out evaluation settings

In [ ]:
print("Run parent directory:", config["output"]["directory"])
held_out_splits = [split for split in ("test", "bolivia") if split in split_paths]
print("Held-out splits evaluated after training:", held_out_splits)
print("Save prediction figures:", eval_cfg.get("save_predictions", False))
print("The next cell starts a fresh training run using the selected YAML.")


#### Train and validate through the shared runner

This cell executes the full epoch budget in the selected YAML. The runner saves the best and last checkpoints, logs, summaries and plots. It then evaluates any configured test/Bolivia splits using the best validation checkpoint. Re-running this cell starts another fresh run.

In [ ]:
run_dir = run_experiment(config_path, device=device)
print("Saved run:", run_dir)


#### Read checkpoint selection and saved held-out results

In [ ]:
summary = json.loads((run_dir / "summary.json").read_text(encoding="utf-8"))
best = summary["best_checkpoint"]
if best is None:
    raise RuntimeError("No best checkpoint was selected; inspect the run log.")
print("Selected epoch:", best["epoch"])
print("Selection split:", best["selection_split"])
print("Selection metric:", best["selection_metric"])
print("Selection value:", best["selection_value"])

reports = {"validation at selected checkpoint": best["validation"]}
for split in ("test", "bolivia"):
    report_path = run_dir / "metrics" / f"{split}.json"
    if report_path.is_file():
        reports[split] = json.loads(report_path.read_text(encoding="utf-8"))
    else:
        print(f"No {split} report in this run; check whether its CSV was configured.")


#### Inspect metrics and saved plots

In [ ]:
for split, report in reports.items():
    print(f"\n{split}")
    print("mIoU:", report["miou"])
    print("Pixel accuracy:", report["pixel_accuracy"])
    print("Per-class IoU:", report["iou_per_class"])
    print("Per-class precision:", report["precision_per_class"])
    print("Per-class recall:", report["recall_per_class"])
    print("Per-class F1:", report["f1_per_class"])
    if "aw_pw_fw" in report:
        for category, values in report["aw_pw_fw"]["summary"].items():
            print(f"{category}: mean per-chip Water IoU={values['mean_chip_iou']}; "
                  f"pooled Water IoU={values['total_iou']}; "
                  f"included chips={values['evaluated_chips']}")

for name in ("training_curves.png", "test_metrics.png", "bolivia_metrics.png"):
    plot_path = run_dir / "plots" / name
    if plot_path.is_file():
        display(Image(filename=str(plot_path)))
